In [13]:
username="Kalisto.Willaey"
cid= "31cb8367284144678c38c3d92bf58fb1"
secret="4511d215e2b24898bdcaaa50e4f4693e"

In [7]:
import pprint
import sys
import spotipy    # la librairie pour manipuler l'api spotify
import spotipy.util as util
import simplejson as json  #pour manipuler les réponses json
import time  #pour créer une playlist horodatée
from datetime import datetime
from random import shuffle #pour attribuer un classement aléatoire aux morceaux

In [9]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from bs4 import BeautifulSoup
import requests

In [14]:
client_credentials_manager = SpotifyClientCredentials(client_id=cid, client_secret=secret)
sp = spotipy.Spotify(client_credentials_manager = client_credentials_manager)

In [26]:
#insert the URI as a string into the function
def get_album_tracks(uri_info):
    uri = []
    track = []
    duration = []
    explicit = []
    track_number = []
    one = sp.album_tracks(uri_info, limit=50, offset=0, market='US')
    df1 = pd.DataFrame(one)
    
    for i, x in df1['items'].items():
        uri.append(x['uri'])
        track.append(x['name'])
        duration.append(x['duration_ms'])
        explicit.append(x['explicit'])
        track_number.append(x['track_number'])
    
    df2 = pd.DataFrame({
    'uri':uri,
    'track':track,
    'duration_ms':duration,
    'explicit':explicit,
    'track_number':track_number})
    
    return df2

In [32]:
#insert output dataframe from the get_album_tracks function
def get_track_info(df):
    danceability = []
    energy = []
    key = []
    loudness = []
    speechiness = []
    acousticness = []
    instrumentalness = []
    liveness = []
    valence = []
    tempo = []
    for i in df['uri']:
        for x in sp.audio_features(tracks=[i]):
            danceability.append(x['danceability'])
            energy.append(x['energy'])
            key.append(x['key'])
            loudness.append(x['loudness'])
            speechiness.append(x['speechiness'])
            acousticness.append(x['acousticness'])
            instrumentalness.append(x['instrumentalness'])
            liveness.append(x['liveness'])
            valence.append(x['valence'])
            tempo.append(x['tempo'])
            
    df2 = pd.DataFrame({
    'danceability':danceability,
    'energy':energy,
    'key':key,
    'loudness':loudness,
    'speechiness':speechiness,
    'acousticness':acousticness,
    'instrumentalness':instrumentalness,
    'liveness':liveness,
    'valence':valence,
    'tempo':tempo})
    
    return df2

In [33]:
def merge_frames(df1, df2):
    df3 = df1.merge(df2, left_index= True, right_index= True)
    return df3

In [34]:
#function to scrape lyrics from genius
def scrape_lyrics(artistname, songname):
    artistname2 = str(artistname.replace(' ','-')) if ' ' in artistname else str(artistname)
    songname2 = str(songname.replace(' ','-')) if ' ' in songname else str(songname)
    page = requests.get('https://genius.com/'+ artistname2 + '-' + songname2 + '-' + 'lyrics')
    html = BeautifulSoup(page.text, 'html.parser')
    lyrics1 = html.find("div", class_="lyrics")
    lyrics2 = html.find("div", class_="Lyrics__Container-sc-1ynbvzw-2 jgQsqn")
    if lyrics1:
        lyrics = lyrics1.get_text()
    elif lyrics2:
        lyrics = lyrics2.get_text()
    elif lyrics1 == lyrics2 == None:
        lyrics = None
    return lyrics

#function to attach lyrics onto data frame
#artist_name should be inserted as a string
def lyrics_onto_frame(df1, artist_name):
    for i,x in enumerate(df1['track']):
        test = scrape_lyrics(artist_name, x)
        df1.loc[i, 'lyrics'] = test
    return df1

In [38]:
coeur_blanc_df1_tracks = get_album_tracks('spotify:album:5IGzOCeKvbUR4q31ZkNz8k')
coeur_blanc_df2_metadata = get_track_info(coeur_blanc_df1_tracks)
df1 = merge_frames(coeur_blanc_df1_tracks, coeur_blanc_df2_metadata)
lyrics_onto_frame(df1, 'Jul')

,uri,track,duration_ms,explicit,track_number,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,lyrics
0,spotify:track:3HrX1V2CpLmlAu2cGjgrjW,Namek,186306,True,1,0.640,0.541,7,-7.731,0.1620,0.75500,0.000445,0.0966,0.765,88.127,NaN
1,spotify:track:6iFjnKoGLAuC24Doa7Z5b4,Le 100 le 100,182586,True,2,0.743,0.793,6,-5.197,0.0657,0.40400,0.012900,0.0995,0.884,140.009,NaN
2,spotify:track:5Wgw0xELVgMedzUXhAxSui,Canette dans les mains,192320,True,3,0.530,0.571,2,-8.004,0.4000,0.80900,0.000012,0.1150,0.504,199.560,NaN
3,spotify:track:27G9K6OFL17tQc14q5EUEA,Chocolata,172840,True,4,0.946,0.704,11,-6.940,0.0798,0.40700,0.001390,0.1260,0.593,124.989,NaN
4,spotify:track:5lxIW1YijMLio4srCKxch2,Forbidden,181466,True,5,0.900,0.527,7,-8.316,0.0485,0.16000,0.002290,0.0943,0.955,130.016,NaN
5,spotify:track:2rVR5Y7yMrMnusm3i3vqTZ,Konami,184666,True,6,0.836,0.774,6,-7.665,0.0728,0.20200,0.001330,0.1060,0.713,143.009,NaN
6,spotify:track:5DQQfgrqgRpkUO6HXtx1jK,Grazie la zone,218586,True,7,0.839,0.748,8,-6.386,0.1650,0.16600,0.000003,0.0750,0.708,133.024,NaN
7,spotify:track:5bXv7jlAxaHsZJ5fshf5Xe,Eh ça va Guy,177306,True,8,0.897,0.823,5,-5.109,0.0936,0.62500,0.001620,0.1180,0.786,104.985,NaN
8,spotify:track:33AXuuLcrOwybbiZ6hcCQy,Héros,238773,True,9,0.886,0.459,9,-7.566,0.2450,0.73100,0.000000,0.1100,0.541,97.906,NaN
9,spotify:track:28sz0xiDFDcEXHdVjnv1ZJ,Pistolet 45,208946,True,10,0.724,0.641,4,-8.693,0.2070,0.29100,0.000002,0.1000,0.438,156.523,NaN
